# 02 · Gold standards & evaluation

*Day 2 — Linguistic Data Analysis II*

By the end you can: **load a gold standard → run an LLM with a fixed prompt → read precision / recall / F1 + a confusion matrix → inspect the errors.** The task (CEFR sentence level, A1–C2) is easy to judge on purpose — today the *mechanics* are the lesson, not the labeling.

You only edit the cells marked **✏️ YOU EDIT**. The `🔧 Library cell` blocks are pre-written — run them, don't change them. (Double-click a library cell if you're curious what's inside.)

## Setup — run this first

In [ ]:
#@title 📦 Setup — run me first { display-mode: "form" }
# Imports + the gold-file location. No pip install needed in Colab.
import json, re, urllib.request, os
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt

def _make_local_backend():
    """Not in Colab → pick a provider by whichever API key is set.
    Add a provider = add a block. (See the 'run locally' guide.)"""
    model = os.environ.get("LLM_MODEL")  # optional override
    if os.environ.get("GEMINI_API_KEY"):
        from google import genai
        client = genai.Client()
        m = model or "gemini-2.5-flash"
        return (lambda p: client.models.generate_content(model=m, contents=p).text,
                f"Gemini API ({m})")
    if os.environ.get("ANTHROPIC_API_KEY"):
        import anthropic
        client = anthropic.Anthropic()
        m = model or "claude-haiku-4-5"
        return (lambda p: client.messages.create(
                    model=m, max_tokens=64,
                    messages=[{"role": "user", "content": p}]).content[0].text,
                f"Anthropic ({m})")
    if os.environ.get("OPENAI_API_KEY"):
        from openai import OpenAI
        client = OpenAI()
        m = model or "gpt-4o-mini"
        return (lambda p: client.chat.completions.create(
                    model=m, messages=[{"role": "user", "content": p}]
                ).choices[0].message.content,
                f"OpenAI ({m})")
    raise RuntimeError(
        "No LLM backend found. Run this notebook in Google Colab (free built-in "
        "Gemini, no key needed), or — to run locally — set one of GEMINI_API_KEY / "
        "ANTHROPIC_API_KEY / OPENAI_API_KEY. See the 'Running locally' guide.")

# LLM backend: Colab's built-in Gemini if available, else a local API.
try:
    from google.colab import ai            # Colab's built-in Gemini — no API key
    generate_text, _backend = (lambda p: ai.generate_text(p)), "Colab Gemini"
except ImportError:
    generate_text, _backend = _make_local_backend()

# CEFR-SP gold set (72 sentences, 12 per level), fetched from the course repo.
GOLD_URL = "https://raw.githubusercontent.com/egumasa/linguistic-data-analysis-II-2026/main/sources/resources/datasets/gold/cefr_sentences.json"
LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
print(f"Setup done. LLM backend: {_backend}. scikit-learn ready.")

In [ ]:
#@title 🔧 Library cell: load_gold(url_or_path) → gold { display-mode: "form" }
def load_gold(url_or_path):
    """Read the canonical gold JSON: [{'id','text','label'}, ...]."""
    if str(url_or_path).startswith("http"):
        raw = urllib.request.urlopen(url_or_path).read().decode("utf-8")
        gold = json.loads(raw)
    else:
        gold = json.loads(open(url_or_path, encoding="utf-8").read())
    print(f"Loaded {len(gold)} items. First one:", gold[0])
    return gold

In [ ]:
#@title 🔧 Library cell: run_prompt(prompt, gold) → predictions { display-mode: "form" }
def _extract_level(text):
    """Pull the first A1/A2/B1/B2/C1/C2 out of the model's reply."""
    m = re.search(r"\b([ABC][12])\b", str(text).upper())
    return m.group(1) if m else "??"

def run_prompt(prompt, gold):
    """Send each item's `text` to the LLM via {text}, collect predicted labels."""
    predictions = []
    for i, item in enumerate(gold, 1):
        reply = generate_text(prompt.format(text=item["text"]))
        predictions.append(_extract_level(reply))
        if i % 12 == 0:
            print(f"  ...{i}/{len(gold)} done")
    print(f"Got {len(predictions)} predictions.")
    return predictions

In [ ]:
#@title 🔧 Library cell: evaluate(gold, predictions) → P/R/F1 + confusion matrix { display-mode: "form" }
def evaluate(gold, predictions):
    y_true = [item["label"] for item in gold]
    y_pred = predictions
    print(classification_report(y_true, y_pred, labels=LEVELS, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=LEVELS)
    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=LEVELS, yticklabels=LEVELS)
    plt.xlabel("Predicted"); plt.ylabel("Gold"); plt.title("Confusion matrix")
    plt.tight_layout(); plt.show()

In [ ]:
#@title 🔧 Library cell: show_errors(gold, predictions) → misclassified table { display-mode: "form" }
def show_errors(gold, predictions):
    rows = [{"id": g["id"], "gold": g["label"], "pred": p, "text": g["text"]}
            for g, p in zip(gold, predictions) if g["label"] != p]
    print(f"{len(rows)} of {len(gold)} wrong.")
    return pd.DataFrame(rows)

## Step 1 — Load the gold standard

Notice the shape: every dataset this week is `{"id", "text", "label"}`. That is the *only* data shape you have to learn.

In [ ]:
gold = load_gold(GOLD_URL)

## Step 2 — Run a fixed prompt   ✏️ YOU EDIT

This is the whole pipeline in one line. Edit the prompt text if you like, then run. `{text}` is where each sentence gets slotted in.

In [ ]:
PROMPT = """You are an expert rater of English sentence difficulty using the CEFR scale.
Classify the sentence into exactly one of: A1, A2, B1, B2, C1, C2.
Answer with the level only.

Sentence: {text}"""

predictions = run_prompt(PROMPT, gold)

## Step 3 — Read the evaluation

Per-level precision / recall / F1 (plus a macro average), then the confusion matrix. For CEFR you'll usually see confusions between *adjacent* levels (B1 ↔ B2), rarely far-apart ones (A1 ↔ C2) — the model has the right idea, imprecise thresholds.

In [ ]:
evaluate(gold, predictions)

## Step 4 — Error analysis

For each miss, ask: is the **gold** defensible, or is this a genuinely borderline sentence? *"Is the disagreement the model's fault or the scheme's?"* is the heart of annotation work — and it returns with force in the Day 3 discourse task.

In [ ]:
show_errors(gold, predictions)

---
You now know the full loop: **load gold → format prompt → call model → evaluate → inspect errors.** On Day 3 the loop is identical; only the *task* gets harder.

::: {.callout-tip}
## Go deeper: code the metrics yourself
`evaluate()` printed precision, recall, F1 and Cohen's κ for you. Before the final project, implement those formulas by hand in the [Coding the metrics from scratch](../code-examples/python/index.md) exercise, then check your work against scikit-learn.
:::